# L1b · Static INT8, second try: first and last layers kept in FP32 (CPU, no GPU)

Follow-up to task **L1** (`l1-edge-benchmark.ipynb`, kept exactly as it ran). Goal **O4**. In L1, static INT8 was the fastest version (1.8 s per image, 2.57× PyTorch), but its Destroyed IoU dropped by 0.0104, just over the 0.01 limit, so it was marked not usable.

**What changes:** the same R9 export and the same static INT8 recipe as L1 (QDQ, per-channel weights, activation ranges from 100 *training* images), except that a few convolutions stay in FP32. Two versions are tried in one run:
- **keep 1+1:** the first convolution (the encoder's stem, which sees the raw image) and the last one (the segmentation head, which writes the class scores);
- **keep 1+3:** the same, plus the two convolutions of the last decoder block, which work at full resolution, where building edges are decided.

**What it measures:**
- mIoU and Destroyed IoU on all 645 validation images (standard prediction, no TTA);
- milliseconds per image on 50 images, with ONNX FP32 timed in the same session as the speed baseline.

The accuracy baseline is L1's FP32 result: PyTorch FP32 and ONNX FP32 both gave 0.5373 / 0.2921 with 100% pixel agreement, so FP32 accuracy isn't measured again. The RAM column is dropped: L1's RAM figures were not reliable.

**How to run on Kaggle:** as L1.
1. Settings: **CPU** (no accelerator), internet ON. Secret `HF_TOKEN` (R9's private repo), or a dataset holding `R9/final_model.pth`.
2. Run with `SMOKE_TEST = True` first (20 validation images, about 15 minutes). Then set it to `False` and use Save & Run All (about 1.5 hours).
3. Outputs in `/kaggle/working/L1b/`: `int8_retry.csv`, `README.md`, `config.json`, `side_by_side.png` and the two static INT8 `.onnx` files. The FP32 `.onnx` is deleted at the end: it is the same file as L1's.

**Done when:** a static INT8 version drops Destroyed IoU by less than 0.01. The fastest one that passes becomes the edge model. If neither passes, ONNX FP32 (1.7× PyTorch, no accuracy loss) stays the edge model.

In [1]:
# ── Cell 1: Install the export and quantisation tools (same as L1) ─────────
import importlib.metadata as _md
import subprocess
import sys

PINNED = {"segmentation-models-pytorch": "0.3.4"}     # same version as the D4 notebooks (checkpoint key layout)
EXTRA = ["onnx", "onnxruntime", "onnxscript"]           # onnxscript: needed by the newer torch ONNX exporter

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

missing = [f"{p}=={v}" for p, v in PINNED.items() if _version(p) != v] + [p for p in EXTRA if _version(p) is None]
if missing:
    print("⏳ Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import onnx
import onnxruntime as ort
import torch
print(f"✓ torch {torch.__version__}, onnx {onnx.__version__}, onnxruntime {ort.__version__}, "
      f"python {sys.version.split()[0]}")

⏳ Installing: segmentation-models-pytorch==0.3.4, onnxruntime, onnxscript
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 7.4 MB/s eta 0:00:00
✓ torch 2.11.0+cpu, onnx 1.22.0, onnxruntime 1.31.0, python 3.13.15


In [2]:
# ── Cell 2: Configuration, checkpoint and the machine ─────────────────────
import glob
import json
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import psutil
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/L1b")

NUM_CLASSES = 4
SMOKE_TEST = False             # True = 20 validation, 10 calibration and 5 latency images
MODEL_NAME, HF_USER = "R9", "AbrarAlam"
CKPT_PATH = None               # set a path to skip the search, e.g. "/kaggle/input/my-ckpts/R9/final_model.pth"
INPUT_SIZE = 1024              # every validation image is 1024 x 1024 (checked); the ONNX input is fixed
OPSET = 17
N_CALIBRATION = 100            # training images for static INT8 calibration
N_LATENCY = 50                 # validation images for the latency runs
LATENCY_REPEATS = 3            # timed runs per image after a warm-up; the median is kept
# The static INT8 versions to try: (first convolutions, last convolutions) kept in FP32
STATIC_KEEP_FP32 = {"INT8 static, keep 1+1 FP32": (1, 1), "INT8 static, keep 1+3 FP32": (1, 3)}
BASELINE = {"miou": 0.5373, "destroyed_iou": 0.2921}   # L1: PyTorch FP32 = ONNX FP32 (100% pixel agreement)
L1_STATIC = {"miou": 0.5251, "destroyed_iou": 0.2817, "drop": 0.0104}   # L1's static INT8 with every layer in INT8
MAX_DROP = 0.01                # INT8 counts as usable if Destroyed IoU drops by less than this

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """Order: CKPT_PATH, then a /kaggle/input folder named R9 / d4_R9 / R9-results, then Hugging Face."""
    if CKPT_PATH:
        return CKPT_PATH
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo, errors = f"{HF_USER}/disasterm3-d4-{name}", []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}.")

def cpu_name():
    try:
        with open("/proc/cpuinfo") as f:
            return next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        return platform.processor() or platform.machine()

CKPT = find_checkpoint(MODEL_NAME)
OUT_DIR.mkdir(parents=True, exist_ok=True)
THREADS = os.cpu_count()
torch.set_num_threads(THREADS)
MACHINE = {"cpu": cpu_name(), "threads": THREADS, "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
           "os": platform.platform(), "torch_threads": torch.get_num_threads()}
print(f"✓ L1b static INT8 retry{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_NAME} from {CKPT}")
print(f"  Machine: {MACHINE}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ L1b static INT8 retry: R9 from /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R9/snapshots/34897e1aa70742cac62a379561e1f969dccd4e10/final_model.pth
  Machine: {'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'threads': 4, 'ram_gb': 33.7, 'os': 'Linux-6.18.48+-x86_64-with-glibc2.39', 'torch_threads': 4}


In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from the D4 notebooks) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split (identical to every D4 run), calibration images and preprocessing ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
calib_pairs = random.Random(0).sample(train_pairs, N_CALIBRATION)   # training images only, never validation
if SMOKE_TEST:
    val_pairs, calib_pairs = val_pairs[:20], calib_pairs[:10]
latency_pairs = val_pairs[:5 if SMOKE_TEST else N_LATENCY]

# The D4 notebooks' A.Normalize (ImageNet mean and std, max_pixel_value 255), without albumentations
MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def preprocess(path):
    """Image file -> float32 array of shape 1 x 3 x H x W, normalised exactly as in training."""
    image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB).astype(np.float32)
    return np.ascontiguousarray(((image - MEAN) / STD).transpose(2, 0, 1)[None])

sizes = Counter(cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE).shape for p in val_pairs)
assert set(sizes) == {(INPUT_SIZE, INPUT_SIZE)}, f"The fixed ONNX input needs {INPUT_SIZE} px images, found {sizes}"
print(f"✓ Validation images: {len(val_pairs)} (expected 645 when SMOKE_TEST is False); "
      f"calibration: {len(calib_pairs)} training images (the same 100 as L1); latency: {len(latency_pairs)} images")

✓ Validation images: 645 (expected 645 when SMOKE_TEST is False); calibration: 100 training images (the same 100 as L1); latency: 50 images


In [5]:
# ── Cell 5: Export to ONNX FP32, then static INT8 with a few convolutions kept in FP32 ──
import segmentation_models_pytorch as smp
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process

def load_pytorch():
    """D4 checkpoints: U-Net + ResNet-50, 4 classes. Accepts a state dict or a resume checkpoint."""
    state = torch.load(CKPT, map_location="cpu", weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.eval()

PATHS = {"ONNX FP32": OUT_DIR / "r9_fp32.onnx"}   # the speed baseline: the same export as L1
prepared = OUT_DIR / "r9_fp32_prepared.onnx"     # shape-inferred and optimised copy that the quantiser starts from

t0 = time.time()
model = load_pytorch()
dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
export_args = dict(input_names=["image"], output_names=["logits"], opset_version=OPSET)
try:
    torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=False, **export_args)
except Exception as e:                            # newer torch may drop the classic exporter
    print(f"Classic exporter failed ({type(e).__name__}: {str(e)[:100]}); using the dynamo exporter")
    torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=True, **export_args)
onnx.checker.check_model(str(PATHS["ONNX FP32"]))
del model
quant_pre_process(str(PATHS["ONNX FP32"]), str(prepared))

# The convolutions in graph order: the first is the encoder's stem, the last is the segmentation head
convs = [n.name for n in onnx.load(str(prepared)).graph.node if n.op_type == "Conv"]
print(f"{len(convs)} convolutions; first {convs[0]!r}, last {convs[-1]!r}")

class TrainingImages(CalibrationDataReader):
    """Feeds the calibration images one by one, so static INT8 learns realistic activation ranges."""
    def __init__(self, pairs):
        self.items = iter(pairs)
    def get_next(self):
        p = next(self.items, None)
        return None if p is None else {"image": preprocess(p["image_path"])}

# The calibrator reduces every tensor to its min and max inside the graph, so 100 images need little RAM.
# Don't set extra_options={"CalibMaxIntermediateOutputs": ...}: in onnxruntime 1.31 it clears the collected
# data without merging it, and calibration then fails with "No data is collected" (the first L1 run, 18 Oct).
EXCLUDED = {}
for kind, (n_first, n_last) in STATIC_KEEP_FP32.items():
    EXCLUDED[kind] = convs[:n_first] + convs[-n_last:]
    PATHS[kind] = OUT_DIR / f"r9_int8_static_keep{n_first}_{n_last}.onnx"
    quantize_static(str(prepared), str(PATHS[kind]), TrainingImages(calib_pairs),
                    quant_format=QuantFormat.QDQ, per_channel=True,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8,
                    nodes_to_exclude=EXCLUDED[kind])
    print(f"  {kind}: FP32 kept for {EXCLUDED[kind]}")

FILE_MB = {k: os.path.getsize(p) / 1e6 for k, p in PATHS.items()}
print(f"✓ Export and quantisation done in {(time.time() - t0) / 60:.1f} min")
for k, mb in FILE_MB.items():
    print(f"  {k:28s} {mb:7.1f} MB")

/tmp/ipykernel_17/2789765085.py:23: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=False, **export_args)
/usr/local/lib/python3.13/dist-packages/segmentation_models_pytorch/base/model.py:17: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if h % output_stride != 0 or w % output_stride != 0:


64 convolutions; first '/encoder/conv1/Conv', last '/segmentation_head/segmentation_head.0/Conv'
  INT8 static, keep 1+1 FP32: FP32 kept for ['/encoder/conv1/Conv', '/segmentation_head/segmentation_head.0/Conv']
  INT8 static, keep 1+3 FP32: FP32 kept for ['/encoder/conv1/Conv', '/decoder/blocks.4/conv1/conv1.0/Conv', '/decoder/blocks.4/conv2/conv2.0/Conv', '/segmentation_head/segmentation_head.0/Conv']
✓ Export and quantisation done in 14.9 min
  ONNX FP32                      130.0 MB
  INT8 static, keep 1+1 FP32      33.1 MB
  INT8 static, keep 1+3 FP32      33.1 MB


## Measure: accuracy of the two static versions, then speed next to ONNX FP32
- **Accuracy:** the standard prediction (no TTA) and the dataset-level confusion matrix, as in L1 and the D4 notebooks. The drop is measured against L1's FP32 Destroyed IoU, 0.2921.
- **Speed:** ONNX FP32 and both static versions are timed in this session, so the speedups compare like with like. A warm-up, then 3 timed runs per image on the first 50 validation images; the median per image is kept.

In [6]:
# ── Cell 6: Accuracy of each static INT8 version on the validation images ──
import gc
import pandas as pd
from tqdm.auto import tqdm

STATIC = list(STATIC_KEEP_FP32)

def make_runner(kind):
    """Image array (1 x 3 x H x W) -> logits array, for one ONNX file."""
    options = ort.SessionOptions()
    options.intra_op_num_threads = THREADS
    session = ort.InferenceSession(str(PATHS[kind]), options, providers=["CPUExecutionProvider"])
    return lambda x: session.run(None, {"image": x})[0]

def confusion(gt, pred):
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def iou_from(cm):
    tp = np.diag(cm).astype(np.float64)
    return tp / (cm.sum(0) + cm.sum(1) - tp + 1e-8)

examples, accuracy = {}, {}
t0 = time.time()
for kind in STATIC:
    run = make_runner(kind)
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)
    for i, p in enumerate(tqdm(val_pairs, desc=kind)):
        pred = run(preprocess(p["image_path"]))[0].argmax(0).astype(np.uint8)
        cm += confusion(cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE), pred)
        if i == 0:
            examples[kind] = pred
    iou = iou_from(cm)
    accuracy[kind] = {"miou": float(iou.mean()), "destroyed_iou": float(iou[3])}
    print(f"  {kind:28s} mIoU {iou.mean():.4f} | Destroyed IoU {iou[3]:.4f} | "
          f"drop {BASELINE['destroyed_iou'] - iou[3]:.4f} (limit {MAX_DROP}) | "
          f"({(time.time() - t0) / 60:.0f} min so far)")
    del run
    gc.collect()

INT8 static, keep 1+1 FP32:   0%|          | 0/645 [00:00<?, ?it/s]

  INT8 static, keep 1+1 FP32   mIoU 0.5363 | Destroyed IoU 0.2918 | drop 0.0003 (limit 0.01) | (19 min so far)


INT8 static, keep 1+3 FP32:   0%|          | 0/645 [00:00<?, ?it/s]

  INT8 static, keep 1+3 FP32   mIoU 0.5363 | Destroyed IoU 0.2918 | drop 0.0003 (limit 0.01) | (38 min so far)


In [7]:
# ── Cell 7: Speed of ONNX FP32 and both static versions (median of 3 timed runs per image, after a warm-up) ──
inputs = [preprocess(p["image_path"]) for p in latency_pairs]
latency = {}
for kind in ["ONNX FP32", *STATIC]:
    run = make_runner(kind)
    run(inputs[0])                                   # warm-up (first call allocates and optimises)
    per_image = []
    for x in tqdm(inputs, desc=kind):
        times = []
        for _ in range(LATENCY_REPEATS):
            t = time.perf_counter()
            run(x)
            times.append(time.perf_counter() - t)
        per_image.append(1000 * float(np.median(times)))
    latency[kind] = {"ms_median": float(np.median(per_image)), "ms_p95": float(np.percentile(per_image, 95))}
    print(f"  {kind:28s} {latency[kind]['ms_median']:8.0f} ms per image (median), "
          f"p95 {latency[kind]['ms_p95']:.0f} ms")
    del run
    gc.collect()

ONNX FP32:   0%|          | 0/50 [00:00<?, ?it/s]

  ONNX FP32                        2289 ms per image (median), p95 2309 ms


INT8 static, keep 1+1 FP32:   0%|          | 0/50 [00:00<?, ?it/s]

  INT8 static, keep 1+1 FP32       1651 ms per image (median), p95 1660 ms


INT8 static, keep 1+3 FP32:   0%|          | 0/50 [00:00<?, ?it/s]

  INT8 static, keep 1+3 FP32       1601 ms per image (median), p95 1621 ms


In [8]:
# ── Cell 8: The table, the verdict, README, config and a side-by-side picture ──
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

base_ms = latency["ONNX FP32"]["ms_median"]
rows = [{"variant": "ONNX FP32 (accuracy from L1)", "file_mb": round(FILE_MB["ONNX FP32"], 1),
         "ms_per_image": round(base_ms), "ms_p95": round(latency["ONNX FP32"]["ms_p95"]),
         "speedup_vs_onnx_fp32": 1.0, "miou": BASELINE["miou"], "destroyed_iou": BASELINE["destroyed_iou"],
         "destroyed_drop": 0.0, "int8_usable": "", "kept_in_fp32": "everything"}]
for kind in STATIC:
    a, l = accuracy[kind], latency[kind]
    drop = BASELINE["destroyed_iou"] - a["destroyed_iou"]
    rows.append({"variant": kind, "file_mb": round(FILE_MB[kind], 1), "ms_per_image": round(l["ms_median"]),
                 "ms_p95": round(l["ms_p95"]), "speedup_vs_onnx_fp32": round(base_ms / l["ms_median"], 2),
                 "miou": round(a["miou"], 4), "destroyed_iou": round(a["destroyed_iou"], 4),
                 "destroyed_drop": round(drop, 4), "int8_usable": "yes" if drop < MAX_DROP else "no",
                 "kept_in_fp32": ", ".join(EXCLUDED[kind])})
table = pd.DataFrame(rows)
table.to_csv(OUT_DIR / "int8_retry.csv", index=False)
print(table.drop(columns=["kept_in_fp32"]).to_string(index=False))

usable = table[table["int8_usable"] == "yes"].sort_values("ms_per_image")
if len(usable):
    best = usable.iloc[0]
    verdict = (f"Edge model: {best['variant']}, {best['speedup_vs_onnx_fp32']}x the speed of ONNX FP32, "
               f"Destroyed IoU drop {best['destroyed_drop']:.4f} (limit {MAX_DROP}).")
else:
    verdict = (f"Neither static version passes (limit {MAX_DROP}): ONNX FP32 stays the edge model "
               "(1.7x PyTorch in L1, no accuracy loss).")
print(f"\n{verdict}")

# The first validation image: ground truth and each static version (0 background, 1 intact, 2 damaged, 3 destroyed)
colours = np.array([[0, 0, 0], [60, 180, 75], [255, 200, 0], [230, 25, 75]], np.uint8)
panels = [("Ground truth", cv2.imread(val_pairs[0]["mask_path"], cv2.IMREAD_GRAYSCALE))] + \
         [(kind, examples[kind]) for kind in STATIC]
fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.4))
for ax, (title, m) in zip(axes, panels):
    ax.imshow(colours[m])
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(Path(val_pairs[0]["image_path"]).name)
fig.tight_layout()
fig.savefig(OUT_DIR / "side_by_side.png", dpi=90)
plt.close(fig)

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

prepared.unlink(missing_ok=True)     # an intermediate copy
PATHS["ONNX FP32"].unlink(missing_ok=True)   # the same file as L1's r9_fp32.onnx; only needed for timing
config = {"task": "L1b", "smoke_test": SMOKE_TEST, "model": MODEL_NAME, "checkpoint": str(CKPT), "machine": MACHINE,
          "input_size": INPUT_SIZE, "opset": OPSET, "calibration_images": len(calib_pairs),
          "validation_images": len(val_pairs), "latency_images": len(latency_pairs),
          "latency_repeats": LATENCY_REPEATS, "kept_in_fp32": EXCLUDED, "baseline_from_L1": BASELINE,
          "onnxruntime": ort.__version__, "torch": torch.__version__}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# L1b: static INT8, second try", "",
    f"Machine: {MACHINE['cpu']}, {MACHINE['threads']} threads, {MACHINE['ram_gb']} GB RAM ({MACHINE['os']}). "
    f"onnxruntime {ort.__version__}, torch {torch.__version__}.", "",
    "Same R9 export and static INT8 recipe as L1 (QDQ, per-channel weights, ranges from 100 training images), "
    "with a few convolutions kept in FP32. Accuracy on "
    f"{len(val_pairs)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''} (standard prediction, no TTA); "
    f"speed as the median of {LATENCY_REPEATS} timed runs per image on {len(latency_pairs)} images, with ONNX FP32 "
    "timed in the same session. FP32 accuracy comes from L1 (PyTorch FP32 = ONNX FP32).", "",
    as_markdown(table.drop(columns=["kept_in_fp32"])), "",
    f"**Verdict:** {verdict}", "",
    f"For comparison, L1's static INT8 with every layer in INT8: mIoU {L1_STATIC['miou']}, Destroyed IoU "
    f"{L1_STATIC['destroyed_iou']}, drop {L1_STATIC['drop']}, so not usable.", "",
    "Convolutions kept in FP32:", "",
    *[f"- {kind}: {', '.join(names)}" for kind, names in EXCLUDED.items()], "",
    "![Side by side](side_by_side.png)", "",
    "Files: `int8_retry.csv`, the two static INT8 `.onnx` files, `config.json`. "
    "`r9_fp32.onnx` is not repeated here: it is the same file as L1's.",
]), encoding="utf-8")
print(f"✓ Wrote int8_retry.csv, side_by_side.png, README.md and config.json to {OUT_DIR}")

                     variant  file_mb  ms_per_image  ms_p95  speedup_vs_onnx_fp32   miou  destroyed_iou  destroyed_drop int8_usable
ONNX FP32 (accuracy from L1)    130.0          2289    2309                  1.00 0.5373         0.2921          0.0000            
  INT8 static, keep 1+1 FP32     33.1          1651    1660                  1.39 0.5363         0.2918          0.0003         yes
  INT8 static, keep 1+3 FP32     33.1          1601    1621                  1.43 0.5363         0.2918          0.0003         yes

Edge model: INT8 static, keep 1+3 FP32, 1.43x the speed of ONNX FP32, Destroyed IoU drop 0.0003 (limit 0.01).
✓ Wrote int8_retry.csv, side_by_side.png, README.md and config.json to /kaggle/working/L1b
